# Dendritic Branch Interpretability

Repeats the somatic analyses but for **dendritic branch activations** — the outputs
of each branch layer *before* they collapse to the soma.

For the **2x2** morphology: each neuron has 4 tip branches (layer 0) → 2 intermediate branches (layer 1) → 1 soma.
For the **3x3x3** morphology: each neuron has 27 tip branches → 9 → 3 → 1 soma.

We hook into each `DendriticBranchLayer` to capture branch activations on the val set.

Analyses:
1. Branch lifetime sparseness — are individual branches class-selective?
2. Per-class mean branch activation raster — does the diagonal appear at the branch level?
3. Branch selectivity vs neuron selectivity — does selectivity increase from tips → soma?
4. Ablation-style: how much does zeroing a branch hurt a class (offline, using saved soma acts)?

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
import sys, torch
from torch.utils.data import DataLoader

sys.path.insert(0, '/n/home13/adithyavm/dendritic-modeling/src')

INTERP   = Path('/n/holylfs06/LABS/kempner_project_b/Lab/dendritic/adivm/dendritic-modeling/outputs/interpretability_results')
OUTPUTS  = Path('/n/holylfs06/LABS/kempner_project_b/Lab/dendritic/adivm/dendritic-modeling/outputs')
NB_DIR   = Path('/n/home13/adithyavm/dendritic-modeling/src/dendritic_modeling/Adi/scripts/interpretability_analysis')
FIG_DIR  = NB_DIR / 'figures_dendrites'
FIG_DIR.mkdir(exist_ok=True)

IMAGENET  = '/n/holylfs06/LABS/kempner_shared/Everyone/testbed/vision/imagenet_1k'
N_CLASSES = 1000
device    = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'device: {device}')
print(f'figures → {FIG_DIR}')

## Load model and register branch hooks

Uses the same fc7-only trained models as `bernardo_meeting_figs.ipynb`:
- 2x2:   `alexnet_k1_ne4096_noproj_ddp` (replaces classifier.4→classifier.5)
- 3x3x3: `alexnet_k1_ne4096_3x3x3_noproj_ddp`

We collect branch layers from `ei_layer.excitatory_cells.branch_layers` explicitly —
`model.core_network.branch_layers` would also include inhibitory dendritic branches,
breaking the downstream `len(S) // N_E` indexing.

Index `[0]` = deepest tips (first to receive input), last index = soma layer.
We hook all layers except the soma to capture branch activations.

In [ ]:
import torch._dynamo
from dendritic_modeling.config import load_config
from dendritic_modeling.scripts.script_utils.setup_utils import initialize_model

# ConfigurableEINetwork wraps its forward in torch.compile; run eager so the
# forward hooks in the next cell fire on every batch.
torch._dynamo.config.disable = True

# ── experiment to analyse ─────────────────────────────────────────────────────
MORPH   = '2x2'   # change to '3x3x3' to analyse the other morphology

# Which trained runs to read. 'fc7' = the checkpoints behind the fc7_*_ne4096
# arrays used in bernardo_meeting_figs.ipynb (only fc7 replaced);
# 'fc6fc7' = the newer runs with both fc6 and fc7 replaced.
RUN_SET = 'fc7'
TAG     = f'{RUN_SET}_{MORPH}_ne4096'

CONFIG_PATHS = {
    ('fc7',    '2x2'):   '/n/home13/adithyavm/dendritic-modeling/configs/vision/alexnet_k1_ne4096_noproj_ddp.yaml',
    ('fc7',    '3x3x3'): '/n/home13/adithyavm/dendritic-modeling/configs/vision/alexnet_k1_ne4096_3x3x3_noproj_ddp.yaml',
    ('fc6fc7', '2x2'):   '/n/home13/adithyavm/dendritic-modeling/configs/vision/alexnet_fc6fc7_ne4096_2x2_ddp.yaml',
    ('fc6fc7', '3x3x3'): '/n/home13/adithyavm/dendritic-modeling/configs/vision/alexnet_fc6fc7_ne4096_3x3x3_ddp.yaml',
}
RUN_GLOBS = {
    ('fc7',    '2x2'):   'alexnet_k1_ne4096_noproj_ddp_*',
    ('fc7',    '3x3x3'): 'alexnet_k1_ne4096_3x3x3_noproj_ddp_*',
    ('fc6fc7', '2x2'):   'alexnet_fc6fc7_ne4096_2x2_ddp_*',
    ('fc6fc7', '3x3x3'): 'alexnet_fc6fc7_ne4096_3x3x3_ddp_2026062*',
}
config_path = CONFIG_PATHS[(RUN_SET, MORPH)]
run_dir     = sorted(OUTPUTS.glob(RUN_GLOBS[(RUN_SET, MORPH)]))[-1]
print(f'Config : {config_path}')
print(f'Run dir: {run_dir}')

config   = load_config(config_path)
model, _ = initialize_model(config.model)
ckpt     = torch.load(run_dir / 'final_model.pt', map_location='cpu', weights_only=False)
model.load_state_dict(ckpt.get('model_state_dict', ckpt))
model.eval().to(device)
print('Model loaded.')

# Branch layers live on the *excitatory* DendriNet of each EI layer.
# NOTE: core_network.branch_layers interleaves excitatory and inhibitory cells,
# so index the excitatory DendriNet directly instead.
ei_layers = model.core_network.layers
assert len(ei_layers) == 1, f'expected one EI layer, got {len(ei_layers)}'
branch_layers = ei_layers[-1].excitatory_cells.branch_layers
n_layers      = len(branch_layers)   # n_branch_layers + 1; last entry is the soma

print(f'\nTotal excitatory branch layers (including soma): {n_layers}')
for i, bl in enumerate(branch_layers):
    label = 'SOMA' if i == n_layers - 1 else f'branch layer {i}'
    print(f'  [{i}] {label}: n_branches={bl.n_branches}')

In [ ]:
from dendritic_modeling.datasets.standard_datasets import load_imagenet_as_datasets

# ── stream the val set, accumulating per-class activation sums ────────────────
# One forward hook per excitatory branch layer (tips → soma); each fires with
# output shape (batch, n_branches) where n_branches = N_E * branches_per_neuron.
#
# The raw activations are never held in memory: for 3x3x3 the tip layer alone
# would be 50000 x 110592 floats (~22 GB). Every figure below only needs the
# per-class means, so those are accumulated batch by batch instead.

layer_names = [f'layer {i}' for i in range(n_layers - 1)] + ['soma']

sums   = [torch.zeros(bl.n_branches, N_CLASSES) for bl in branch_layers]
counts = torch.zeros(N_CLASSES)
batch_out = {}

handles = [
    bl.register_forward_hook(
        lambda m, inp, out, i=i: batch_out.__setitem__(i, out.detach().float().cpu())
    )
    for i, bl in enumerate(branch_layers)
]

_, _, val_ds = load_imagenet_as_datasets(
    train_valid_split=0.8, flatten=False, normalize=True, task_data_path=IMAGENET)
val_loader = DataLoader(val_ds, batch_size=128, shuffle=False, num_workers=8, pin_memory=True)

with torch.no_grad():
    for batch_idx, (imgs, targets) in enumerate(val_loader):
        batch_out.clear()
        model(imgs.to(device))
        assert len(batch_out) == n_layers, 'a branch-layer hook did not fire'
        counts.index_add_(0, targets, torch.ones(len(targets)))
        for i in range(n_layers):
            sums[i].index_add_(1, targets, batch_out[i].T)
        if (batch_idx + 1) % 50 == 0:
            print(f'  {int(counts.sum())} images...')

for h in handles:
    h.remove()

denom       = counts.clamp(min=1)
branch_pcms = [(s / denom).numpy() for s in sums[:-1]]   # list of (n_branches, 1000)
soma_pcm    = (sums[-1] / denom).numpy()                  # (N_E, 1000)

print(f'\nimages seen: {int(counts.sum())}   classes present: {int((counts > 0).sum())}')
for name, pcm in zip(layer_names, branch_pcms + [soma_pcm]):
    print(f'  {name:>8}: per-class mean activation {pcm.shape}')

In [ ]:
# ── lifetime sparseness from the per-class means ──────────────────────────────

def lifetime_sparseness(pcm, n_classes=N_CLASSES):
    """pcm: (n_units, n_classes) → S (n_units,)   [Vinje & Gallant 2000]"""
    r_pos = np.maximum(pcm, 0)
    n     = n_classes
    num   = (r_pos.sum(axis=1) / n) ** 2
    denom = (r_pos ** 2).sum(axis=1) / n
    S = np.where(denom > 1e-10, (1 - num / denom) / (1 - 1 / n), 0.0)
    return S.astype(np.float32)

print('Computing lifetime sparseness...')
branch_Ss = [lifetime_sparseness(p) for p in branch_pcms]
soma_S    = lifetime_sparseness(soma_pcm)

for name, S in zip(layer_names, branch_Ss + [soma_S]):
    print(f'  {name:>8}: S mean={S.mean():.4f}  dead={(S == 0).sum()}/{len(S)}')

## Figure 1 — Lifetime sparseness distribution: tips → soma

Does selectivity increase as we move from dendritic tips toward the soma?

In [ ]:
all_layers = branch_Ss + [soma_S]
layer_labels = [f'layer {i} (tips)' if i == 0 else
                (f'layer {i}' if i < len(branch_Ss) else 'soma')
                for i in range(len(all_layers))]

fig, axes = plt.subplots(1, len(all_layers), figsize=(5*len(all_layers), 4), sharey=False)
if len(all_layers) == 1: axes = [axes]

cmap = plt.cm.viridis
colors = [cmap(i/(len(all_layers)-1)) for i in range(len(all_layers))]

for ax, S, lbl, c in zip(axes, all_layers, layer_labels, colors):
    ax.hist(S, bins=80, color=c, edgecolor='none', alpha=0.85)
    ax.axvline(S.mean(), color='#333', lw=1.2, ls='--', label=f'mean={S.mean():.4f}')
    ax.set_xlabel('lifetime sparseness S', fontsize=10)
    ax.set_ylabel('# units', fontsize=10)
    ax.set_title(f'{lbl}\nn={len(S)}', fontsize=10)
    ax.legend(fontsize=8)
    ax.spines[['top','right']].set_visible(False)

plt.suptitle(f'{MORPH}: lifetime sparseness from tips to soma\n'
             'Does selectivity increase toward the soma?', fontsize=12, y=1.02)
plt.tight_layout()
plt.savefig(FIG_DIR / f'fig_branch_sparseness_{MORPH}.pdf', bbox_inches='tight')
plt.show()

print('\nMean S by layer (tips → soma):')
for lbl, S in zip(layer_labels, all_layers):
    print(f'  {lbl}: {S.mean():.5f}')

## Figure 2 — Raster map for branch activations

Same as the somatic raster but for each branch layer.
Branches sorted by their preferred class. Diagonal = class-selective branches.

In [ ]:
all_pcms   = branch_pcms + [soma_pcm]
all_labels_raster = [f'layer {i}' for i in range(len(branch_pcms))] + ['soma']

fig, axes = plt.subplots(1, len(all_pcms), figsize=(6*len(all_pcms), 5))
if len(all_pcms) == 1: axes = [axes]

for ax, pcm, lbl in zip(axes, all_pcms, all_labels_raster):
    order      = np.argsort(pcm.argmax(axis=1))
    subsample  = max(1, len(order) // 1024)   # show ~1024 units max
    pcm_sorted = pcm[order][::subsample]
    vmax = np.percentile(pcm_sorted, 99)
    im = ax.imshow(pcm_sorted, aspect='auto', cmap='inferno',
                   vmin=0, vmax=vmax, interpolation='nearest')
    ax.set_xlabel('ImageNet class', fontsize=9)
    ax.set_ylabel(f'Unit (sorted by pref. class)', fontsize=9)
    ax.set_title(f'{lbl}  n={pcm.shape[0]}', fontsize=10)
    plt.colorbar(im, ax=ax, fraction=0.015, pad=0.02, label='mean act')

plt.suptitle(f'{MORPH}: per-class mean activation raster — tips to soma\n'
             'Diagonal = class-selective units', fontsize=12, y=1.02)
plt.tight_layout()
plt.savefig(FIG_DIR / f'fig_branch_raster_{MORPH}.pdf', bbox_inches='tight')
plt.show()

## Figure 3 — Mean S across layers (tips → soma)

Single summary plot: does selectivity monotonically increase as signals propagate toward soma?

In [ ]:
means   = [S.mean() for S in all_layers]
medians = [np.median(S[S > 0]) if (S > 0).sum() > 0 else 0 for S in all_layers]
xlabels = layer_labels

fig, ax = plt.subplots(figsize=(7, 4))
x = np.arange(len(means))
ax.plot(x, means,   'o-', color='#2a78d6', lw=2, ms=7, label='mean S (all units)')
ax.plot(x, medians, 's--', color='#e07b39', lw=1.5, ms=6, label='median S (active only)')
ax.set_xticks(x); ax.set_xticklabels(xlabels, rotation=20, ha='right')
ax.set_ylabel('lifetime sparseness S', fontsize=11)
ax.set_title(f'{MORPH}: does selectivity increase from tips to soma?', fontsize=12)
ax.legend(fontsize=9)
ax.spines[['top','right']].set_visible(False)
plt.tight_layout()
plt.savefig(FIG_DIR / f'fig_S_by_layer_{MORPH}.pdf', bbox_inches='tight')
plt.show()

## Figure 4 — Branch vs soma selectivity per neuron

For each excitatory neuron: scatter its mean branch S (averaged over its own branches) vs its soma S.
Points above the diagonal = soma is MORE selective than its own dendrites (selectivity emerges at soma).
Points below = dendrites already selective, soma loses it.

In [ ]:
N_E = soma_pcm.shape[0]

# For each branch layer, reshape (n_branches_total,) → (N_E, branches_per_neuron) and average S
fig, axes = plt.subplots(1, len(branch_Ss), figsize=(5*len(branch_Ss), 5))
if len(branch_Ss) == 1: axes = [axes]

for ax, (S_branch, lbl) in zip(axes, zip(branch_Ss, layer_labels)):
    n_branches_per_neuron = len(S_branch) // N_E
    if len(S_branch) % N_E != 0:
        print(f'Warning: {lbl} n_branches={len(S_branch)} not divisible by N_E={N_E}')
        ax.set_visible(False)
        continue
    S_branch_per_neuron = S_branch.reshape(N_E, n_branches_per_neuron).mean(axis=1)   # (N_E,)

    ax.scatter(S_branch_per_neuron, soma_S, s=2, alpha=0.3, rasterized=True, color='#8e44ad')
    lim = [0, max(S_branch_per_neuron.max(), soma_S.max()) * 1.05]
    ax.plot(lim, lim, color='#333', lw=1, ls='--', alpha=0.5)
    r = np.corrcoef(S_branch_per_neuron, soma_S)[0,1]
    ax.set_xlabel(f'mean branch S ({lbl})', fontsize=10)
    ax.set_ylabel('soma S', fontsize=10)
    ax.set_title(f'r={r:.3f}  branches/neuron={n_branches_per_neuron}', fontsize=10)
    ax.spines[['top','right']].set_visible(False)

plt.suptitle(f'{MORPH}: branch S vs soma S per neuron\nAbove diagonal = soma more selective than its dendrites',
             fontsize=12, y=1.02)
plt.tight_layout()
plt.savefig(FIG_DIR / f'fig_branch_vs_soma_S_{MORPH}.pdf', bbox_inches='tight')
plt.show()